### Data Sourcing

In [ ]:
import pandas as pd
import numpy as np

path_94_00 = "../data/modified/poverty_incidence/psy_prov_poverty_1994_2000.csv"
path_03_06 = "../data/modified/poverty_incidence/psy_prov_poverty_2003_2006.csv"
path_09_15 = "../data/modified/poverty_incidence/psy_prov_poverty_2009_2015.csv"
path_18_23 = "../data/modified/poverty_incidence/psy_prov_poverty_2018_2023.csv"

# Load the CSVs
df_94_00 = pd.read_csv(path_94_00)
df_03_06 = pd.read_csv(path_03_06)
df_09_15 = pd.read_csv(path_09_15)
df_18_23 = pd.read_csv(path_18_23)

### Dataframe Reshaping

In [ ]:
def process_wide_to_long(df):
    # Forcefully rename the first two columns to destroy any hidden BOM characters
    df.columns = ['reg_name', 'prov_name'] + list(df.columns[2:])
    
    # Identify year columns (columns that are exactly 4 digits)
    year_cols = [col for col in df.columns if str(col).isdigit() and len(str(col)) == 4]
    
    # Isolate Region, Province, and the year columns.
    df_clean = df[['reg_name', 'prov_name'] + year_cols].copy()
    
    # Melt from wide to long using both region and province as IDs
    melted = df_clean.melt(id_vars=['reg_name', 'prov_name'], 
                           value_vars=year_cols, 
                           var_name='year', 
                           value_name='poverty_incidence')
    return melted

# Apply to all dataframes
long_94_00 = process_wide_to_long(df_94_00)
long_03_06 = process_wide_to_long(df_03_06)
long_09_15 = process_wide_to_long(df_09_15)
long_18_23 = process_wide_to_long(df_18_23)

# Concatenate all years into one master dataframe
df_master = pd.concat([long_94_00, long_03_06, long_09_15, long_18_23], ignore_index=True)

# Clean up data types
df_master['year'] = df_master['year'].astype(int)
df_master['poverty_incidence'] = pd.to_numeric(df_master['poverty_incidence'], errors='coerce')

# Clean strings to ensure no hidden spaces cause issues later
df_master['prov_name'] = df_master['prov_name'].astype(str).str.strip()
df_master['reg_name'] = df_master['reg_name'].astype(str).str.strip()

# Drop rows where poverty_incidence is NaN
df_master = df_master.dropna(subset=['poverty_incidence'])

### Time-Series Formatting

In [ ]:
# Sort the data geographically and chronologically
df_master = df_master.sort_values(by=['reg_name', 'prov_name', 'year']).reset_index(drop=True)

# Create the 'time_idx' (for Temporal Fusion Transformers)
unique_years = sorted(df_master['year'].unique())
year_to_idx = {year: idx for idx, year in enumerate(unique_years)}
df_master['time_idx'] = df_master['year'].map(year_to_idx)

# Reorder columns for readability and logical flow
df_master = df_master[['reg_name', 'prov_name', 'year', 'time_idx', 'poverty_incidence']]

### Poverty Dataframe Exporting

In [ ]:
# Export to the processed data folder
output_path = "../data/processed/poverty_incidence_long.csv"
df_master.to_csv(output_path, index=False)

# Quick sanity check output
print(f"Successfully processed and saved {len(df_master)} rows.")
print("Year mapping used for time_idx:", year_to_idx)
display(df_master.head())